# 4-2. AI Agent와 Multi-Agent 패턴 — 이론과 실습

---

## 목차

| # | 내용 |
|:---:|------|
| 0 | **환경 설정** — 패키지 설치, API 키, 모델/도구 초기화 |
| 1 | **Agent란 무엇인가** — 4대 구성요소 (LLM, Tool, Memory, Plan) |
| 2 | **Tool-use** — LLM이 "행동"할 수 있게 만들기 |
| 3 | **Memory** — 대화 맥락 유지 |
| 4 | **ReAct** — 추론과 행동의 반복 (Direct vs ReAct) |
| 5 | **Trustworthiness** — Agent의 신뢰성 확보 (가드레일, 검증) |
| 6 | **Multi-Agent 패턴** — Planner-Worker, Reflection |
| 7 | **정리** |

<br>

> **📌 이 실습은**
>
> 4-1에서 배운 RAG를 기반으로, 이번에는 **Agent가 직접 행동하고 판단하는** 구조를 배운다.
>
> 개념을 이해한 뒤, 자기주도 실습(`실습_4-2(1)`, `실습_4-2(2)`)에서 직접 코드를 작성한다.


---

## 0. 환경 설정


In [ ]:
# 패키지 설치 (최초 1회)
!pip install -q \
    langchain>=1.0.0 langchain-openai>=1.0.0 langchain-upstage>=0.3.0 \
    langchain-community>=0.3.0 langgraph>=1.0.0 langchain-text-splitters>=1.0.0 \
    chromadb>=0.5.0 tiktoken>=0.7.0 python-dotenv>=1.0.0 tokenizers>=0.22.0


In [ ]:
import os
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

load_dotenv()

# Upstage API 사용 시 필요
if not os.environ.get('UPSTAGE_API_KEY'):
    os.environ['UPSTAGE_API_KEY'] = input('UPSTAGE_API_KEY를 입력하세요: ')

# OpenAI API 사용 시 필요 (Option B 선택 시)
# if not os.environ.get('OPENAI_API_KEY'):
#     os.environ['OPENAI_API_KEY'] = input('OPENAI_API_KEY를 입력하세요: ')

print('환경 설정 완료')


In [ ]:
from langchain_openai import ChatOpenAI

# ═══════════════════════════════════════════════════════════
# 모델 초기화 — Solar / OpenAI 중 택 1
# ═══════════════════════════════════════════════════════════

# ── Option A: Solar (기본) ──
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model='solar-pro3')

# ── Option B: OpenAI 대안 (주석 해제하여 사용) ──
# llm = ChatOpenAI(model='gpt-5-nano', temperature=0)

print(f'모델 초기화 완료')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 역할: 실습에서 사용할 시뮬레이션 환경 구성
# - 실제 DB 대신 딕셔너리로 주문 데이터를 구성한다.
# - Agent가 호출할 도구의 원본 함수를 정의한다.
# ═══════════════════════════════════════════════════════════

# ========== 1. 주문 DB (시뮬레이션) ==========
# 실제 서비스에서는 MySQL, PostgreSQL 등의 DB에서 조회한다.
# 실습에서는 딕셔너리로 대체한다.
orders_db = {
    'ORD001': {'status': '배송 지연', 'product': '노트북', 'customer': '홍길동'},
    'ORD002': {'status': '배송 완료', 'product': '키보드', 'customer': '김철수'},
    'ORD003': {'status': '배송 중', 'product': '마우스', 'customer': '이영희'},
}

# 쿠폰 발급 기록 (Agent가 실제로 쿠폰을 발급했는지 확인용)
issued_coupons = {}

# ========== 2. 주문 조회 함수 ==========
# 역할: 주문번호를 받아 해당 주문의 상태, 상품명을 반환한다.
# 이 함수가 나중에 @tool 데코레이터로 감싸져 Agent의 "도구"가 된다.
def get_order_status(order_id: str) -> str:
    """주문 상태를 조회한다."""
    if order_id in orders_db:
        order = orders_db[order_id]
        return f'주문번호: {order_id}, 상태: {order["status"]}, 상품: {order["product"]}'
    return f'주문번호 {order_id}를 찾을 수 없습니다.'

# ========== 3. 쿠폰 발급 함수 ==========
# 역할: 주문번호와 금액을 받아 쿠폰을 발급한다.
# 검증: 한도 초과(20,000원) 시 발급을 거부한다.
# issued_coupons에 기록하여 실제 발급 여부를 확인할 수 있다.
def issue_coupon(order_id: str, amount: int) -> str:
    """쿠폰을 발급한다."""
    if order_id not in orders_db:
        return f'주문번호 {order_id}를 찾을 수 없습니다.'
    if amount > 20000:
        return f'발급 실패: 1회 최대 한도(20,000원) 초과'
    issued_coupons[order_id] = amount  # ← 실제 발급 기록
    return f'쿠폰 발급 완료: 주문 {order_id}에 {amount:,}원 쿠폰 발급'

print(f'주문 데이터 {len(orders_db)}건 준비 완료')


---

## 1. Agent란 무엇인가?

### 1-1. 4-1에서 배운 RAG와의 차이

4-1에서 배운 RAG 파이프라인은 **"검색 → 생성"의 고정된 흐름**이었다.

Agent는 여기서 한 단계 더 나아간다: **LLM이 스스로 판단하여 어떤 행동을 할지 결정**한다.

| 구분 | RAG (4-1) | Agent (4-2) |
|------|:---:|:---:|
| 흐름 | 검색 → 생성 (고정) | **LLM이 상황에 따라 다른 행동 선택** |
| 도구 사용 | 검색(Retriever)만 | 검색, 쿠폰 발급, DB 조회 등 **다양한 도구** |
| 판단 주체 | 개발자가 흐름을 설계 | **LLM이 스스로 판단** |

### 1-2. Agent의 4대 구성요소

![Image E](https://i.ibb.co/6cXZL8pF/image-E.png)

Agent는 네 가지 핵심 요소로 구성된다.

| 구성요소 | 역할 | 비유 |
|---------|------|------|
| **LLM** (두뇌) | 상황을 이해하고 판단 | 사람의 뇌 |
| **Tool** (손) | 실제 행동을 수행 (API 호출, DB 조회 등) | 사람의 손과 도구 |
| **Memory** (기억) | 이전 대화/작업을 기억 | 사람의 기억력 |
| **Plan** (전략) | 복잡한 작업을 단계별로 분해 | 사람의 계획 능력 |

```
┌─────────────────────────────────────────────────┐
│                   Agent                        │
│                                                │
│   [Plan]  → 무엇을 할지 계획                   │
│     ↓                                          │
│   [LLM]   → 상황을 판단하고 결정               │
│     ↓                                          │
│   [Tool]  → 실제 행동 수행 (검색, 발급 등)     │
│     ↓                                          │
│   [Memory] → 결과를 기억하고 다음 판단에 활용  │
│                                                │
└────────────────────────────────────────────────┘
```

다음 챕터부터 이 4가지를 하나씩 추가하며 Agent를 완성해 나간다.


---

## 2. Tool-use — LLM이 "행동"할 수 있게 만들기

![Image A](https://i.ibb.co/8LjMdLcC/image-A.png)

### 2-1. Tool-use(Function Calling)란?

LLM은 본질적으로 **텍스트를 생성하는 모델**이다.
아무리 똑똑하더라도 혼자서는 이메일을 보내거나, DB를 조회하거나, 쿠폰을 발급할 수 없다.

**Tool-use**(도구 사용, Function Calling이라고도 부른다)는
LLM이 **외부 함수(도구)를 직접 호출**하여 실제 세계에 영향을 미칠 수 있게 하는 기능이다.

> **💡 비유: 뇌와 손의 관계**
>
> - **LLM만** = 뇌만 있고 손이 없는 상태. "문을 열어야 한다"고 생각할 수 있지만 실제로 문을 열 수 없다.
> - **LLM + Tool** = 뇌에 손이 연결된 상태. 생각하고 판단한 뒤, 손으로 직접 행동할 수 있다.

### 2-2. Tool-use의 동작 과정 (5단계)

Tool-use는 다음 5단계로 동작한다:

```
① 사용자 요청      "주문 ORD001 상태 확인해줘"
      ↓
② LLM이 분석       "주문 조회 도구를 써야겠군"
      ↓
③ 도구 선택+호출    get_order_status("ORD001")  ← LLM이 직접 결정
      ↓
④ 결과 반환         "배송 지연, 노트북, 홍길동"
      ↓
⑤ 최종 답변 생성    "ORD001 주문은 현재 배송 지연 상태입니다."
```

핵심은 **③번 단계**이다: LLM이 도구의 이름, 설명, 파라미터 정보를 보고
**"어떤 도구를, 어떤 인자로 호출할지" 스스로 판단**한다.

### 2-3. LLM은 어떻게 도구를 "알게" 되는가?

`bind_tools()` 메서드가 이 역할을 한다.
각 도구의 **함수 이름, docstring(설명), 파라미터 타입**을 JSON 스키마로 변환하여 LLM에게 전달한다.

```python
# @tool 데코레이터가 자동 생성하는 스키마 (내부 동작)
{
    "name": "get_order_status_tool",
    "description": "주문 상태를 조회한다. 주문번호를 입력하면 현재 상태를 반환.",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "주문번호"}
        }
    }
}
```

따라서 **docstring을 잘 작성하는 것이 매우 중요**하다.
LLM은 이 설명을 읽고 "이 도구가 내 상황에 맞는지" 판단하기 때문이다.

### 2-4. LLM만으로는 행동할 수 없다 — 직접 확인


In [ ]:
from langchain_core.messages import HumanMessage

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: LLM은 텍스트만 생성할 뿐, 실제 행동을 수행하지 않는다.
# → issued_coupons가 비어 있는 것으로 확인할 수 있다.
# ═══════════════════════════════════════════════════════════

# LLM에게 쿠폰 발급을 "요청"한다.
# llm.invoke(): LLM에게 메시지를 보내고 응답을 받는 핵심 메서드
response = llm.invoke([HumanMessage(content='주문 ORD001에 5000원 쿠폰을 발급해줘.')])
print('LLM 응답:', response.content[:200])

# ★ 핵심 확인: 실제로 쿠폰이 발급되었는가?
# → issued_coupons 딕셔너리를 확인하면 비어 있다.
# → LLM은 "발급했습니다"라고 텍스트를 생성했을 뿐, issue_coupon() 함수를 호출하지 않았다.
print(f'\n실제 발급된 쿠폰: {issued_coupons}')
print('→ 비어 있다! LLM은 텍스트만 생성했을 뿐, 실제로는 아무 일도 일어나지 않았다.')


### 2-5. `@tool` 데코레이터와 `bind_tools()`

코드에서 Tool을 정의하는 두 가지 핵심 메커니즘:

| 요소 | 역할 |
|------|------|
| `@tool` 데코레이터 | 일반 함수를 LangChain Tool로 변환. **docstring이 도구 설명**이 된다 |
| `bind_tools()` | LLM에게 사용 가능한 도구 목록을 알려준다 |
| `create_agent()` | LLM + Tools를 결합하여 자동으로 ReAct Agent를 생성 |


In [ ]:
from langchain_core.tools import tool

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심:
# 1. @tool 데코레이터로 일반 함수를 LangChain Tool로 변환
# 2. bind_tools()로 LLM에게 사용 가능한 도구 목록을 전달
# 3. LLM이 직접 답하지 않고 "이 도구를 호출해줘"라고 요청하는 것을 확인
# ═══════════════════════════════════════════════════════════

# ========== @tool 데코레이터 ==========
# 역할: 일반 파이썬 함수를 LangChain의 Tool 객체로 변환한다.
# ★ 중요: docstring이 LLM에게 "이 도구가 뭘 하는지" 설명하는 역할을 한다.
#   LLM은 이 설명을 읽고 "이 도구가 지금 필요한가?"를 판단한다.
#   따라서 docstring을 명확하게 작성해야 LLM이 올바른 도구를 선택한다.

@tool
def get_order_status_tool(order_id: str) -> str:
    """주문 상태를 조회한다. 주문번호(예: ORD001)를 입력하면 현재 상태를 반환한다."""
    return get_order_status(order_id)

@tool
def issue_coupon_tool(order_id: str, amount: int) -> str:
    """쿠폰을 발급한다. 주문번호와 금액(원)을 입력한다. 1회 최대 20,000원."""
    return issue_coupon(order_id, amount)

tools = [get_order_status_tool, issue_coupon_tool]

# ========== bind_tools() ==========
# 역할: LLM에게 "이런 도구들을 사용할 수 있다"고 알려준다.
# 내부 동작: 각 도구의 이름, 설명, 파라미터 타입을 JSON 스키마로 변환하여
#   LLM 호출 시 함께 전달한다.
llm_with_tools = llm.bind_tools(tools)

# ========== 도구 바인딩된 LLM에게 질문 ==========
response = llm_with_tools.invoke([HumanMessage(content='주문 ORD001의 상태를 확인해줘')])

# ★ 핵심 확인: LLM이 직접 답하지 않고, "이 도구를 호출해달라"고 요청한다.
# response.tool_calls에 호출할 도구 이름과 인자가 담겨 있다.
print('LLM이 요청한 도구 호출:')
for tc in response.tool_calls:
    print(f'  도구: {tc["name"]}, 인자: {tc["args"]}')


In [ ]:
from langchain.agents import create_agent

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: create_agent()
# - LLM + Tools를 결합하여 "도구 선택 → 실행 → 결과 확인"을 자동화한다.
# - 내부적으로 ReAct(Thought→Action→Observation) 순환이 구현되어 있다.
# - 이전 셀의 bind_tools + 도구 실행 + 결과 반환을 한 줄로 해결한다.
# ═══════════════════════════════════════════════════════════

# system prompt: Agent의 역할과 행동 지침을 정의한다.
system_prompt = '당신은 고객 서비스 AI 에이전트입니다. 제공된 도구를 사용하여 고객 요청을 처리하세요.'

# ★ 핵심 코드: create_agent()
# - model: 사용할 LLM
# - tools: Agent가 사용할 수 있는 도구 목록
# - system_prompt: Agent의 시스템 프롬프트
agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

# Agent 실행: invoke()에 messages를 전달하면 자동으로
# 도구 선택 → 실행 → 결과 확인 → 최종 답변 생성까지 처리한다.
result = agent.invoke({
    'messages': [{'role': 'user', 'content': '주문 ORD001의 상태를 확인하고, 배송 지연이면 5000원 쿠폰을 발급해줘.'}]
})

# result['messages'][-1]: 가장 마지막 메시지 = Agent의 최종 응답
print('Agent 응답:')
print(result['messages'][-1].content)

# ★ 핵심 확인: 이번에는 issued_coupons에 실제 기록이 있다!
print(f'\n실제 발급된 쿠폰: {issued_coupons}')
print('→ 이번에는 진짜로 쿠폰이 발급되었다!')


---

## 3. Memory — 대화 맥락 유지

### 3-1. Agent에게 Memory가 필요한 이유

사람은 대화할 때 이전에 나눈 말을 **자연스럽게 기억**한다.
하지만 LLM은 기본적으로 **각 요청을 독립적으로 처리**한다.
즉, 이전 대화 내용을 전혀 기억하지 못한다.

```
[Memory 없는 Agent]
사용자: "내 주문번호는 ORD001이야."
Agent:  "네, ORD001 확인했습니다."

사용자: "그 주문 상태 알려줘."
Agent:  "어떤 주문을 말씀하시는 건가요?"  ← 기억 못함!
```

이는 고객 서비스에서 치명적이다. 고객이 매번 정보를 반복해야 하면 불만이 생긴다.

### 3-2. Memory의 종류

| 종류 | 설명 | 비유 |
|------|------|------|
| **단기 기억** (Working Memory) | 현재 대화 세션 내의 맥락 | 회의 중 메모 |
| **장기 기억** (Long-term Memory) | 세션 간에도 유지되는 정보 | 고객 CRM 데이터 |

이 실습에서는 **단기 기억**에 해당하는 `MemorySaver`를 사용한다.

### 3-3. MemorySaver의 동작 원리

LangGraph의 `MemorySaver`는 **대화 기록을 자동으로 저장하고 복원**한다.

```
[MemorySaver 동작]

대화 1: "내 주문번호는 ORD001이야"
  → MemorySaver가 저장: {thread_id: "user123", messages: [...]}

대화 2: "그 주문 상태 알려줘"
  → MemorySaver가 thread_id="user123"의 이전 대화를 불러옴
  → Agent가 ORD001을 기억하고 상태 조회
```

핵심 파라미터: **`thread_id`** — 같은 ID면 같은 대화 세션으로 취급된다.
고객별로 다른 `thread_id`를 부여하면 각 고객의 대화를 독립적으로 관리할 수 있다.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: MemorySaver + thread_id로 대화 맥락을 유지한다.
# - checkpointer=memory: 대화 기록을 저장/복원하는 객체
# - thread_id: 대화 세션을 구분하는 고유 ID
# ═══════════════════════════════════════════════════════════

# ========== 1. MemorySaver 생성 ==========
# 역할: Agent의 대화 기록을 메모리에 저장한다.
# 실무에서는 SqliteSaver, PostgresSaver 등 영구 저장소를 사용할 수 있다.
memory = MemorySaver()

# ========== 2. Memory가 적용된 Agent 생성 ==========
# ★ 핵심: checkpointer=memory를 전달하면 대화 기록이 자동 저장/복원된다.
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt='당신은 고객 서비스 AI 에이전트입니다. 대화 기록을 참고하여 응답하세요.',
    checkpointer=memory,
)

# ========== 3. thread_id로 대화 세션 구분 ==========
# 같은 thread_id = 같은 대화 세션. 이전 대화를 기억한다.
# 다른 thread_id = 다른 대화 세션. 서로 독립적이다.
# 예: 고객A='user-A', 고객B='user-B'로 구분
config = {'configurable': {'thread_id': 'user123'}}

# 첫 번째 대화: 주문번호를 알려준다
r1 = agent_with_memory.invoke(
    {'messages': [{'role': 'user', 'content': '내 주문번호는 ORD001이야.'}]},
    config=config,  # ← 같은 config(thread_id)를 전달
)
print('응답 1:', r1['messages'][-1].content[:150])

# 두 번째 대화: "그 주문"이라고만 말해도 ORD001을 기억한다
r2 = agent_with_memory.invoke(
    {'messages': [{'role': 'user', 'content': '그 주문 상태 확인해줘.'}]},
    config=config,  # ← 같은 config(thread_id)를 전달해야 기억함
)
print('\n응답 2:', r2['messages'][-1].content[:200])
print('\n→ 이전 대화의 주문번호(ORD001)를 기억하고 정확히 조회한다!')


---

## 4. ReAct — 추론과 행동의 반복

![Image B](https://i.ibb.co/TDM7T8ML/image-B.png)

### 4-1. 복잡한 요청의 문제

단순한 요청("주문 상태 확인해줘")은 도구 1회 호출로 해결된다.
하지만 실제 고객 서비스에서는 **여러 단계의 판단과 행동**이 필요한 경우가 많다.

```
"주문 ORD001, ORD002, ORD003의 상태를 확인하고, 배송 지연인 주문에만 쿠폰을 발급해줘."

→ 이 요청을 처리하려면:
  1. ORD001 상태 조회 → 배송 지연 → 쿠폰 발급 대상
  2. ORD002 상태 조회 → 배송 완료 → 쿠폰 발급 불필요
  3. ORD003 상태 조회 → 배송 중 → 쿠폰 발급 불필요
  4. ORD001에만 쿠폰 발급
  5. 결과를 종합하여 고객에게 답변

→ 최소 4번의 도구 호출 + 매 단계마다 "다음에 뭘 해야 하지?"를 판단해야 한다.
```

이런 문제를 해결하기 위해 등장한 것이 **ReAct** 프레임워크이다.

### 4-2. ReAct (Reasoning + Action)란?

ReAct는 2022년 구글 연구팀이 발표한 프레임워크로,
**추론(Reasoning)과 행동(Action)을 번갈아 수행**하며 문제를 단계적으로 해결한다.

핵심 사이클: **Thought → Action → Observation**

| 단계 | 설명 | 예시 |
|------|------|------|
| **Thought** (추론) | "지금 무엇을 해야 하는가?" 판단 | "ORD001의 상태를 먼저 확인해야 한다" |
| **Action** (행동) | 판단에 따라 도구를 호출 | `get_order_status("ORD001")` |
| **Observation** (관찰) | 도구 실행 결과를 확인 | "배송 지연" |

이 사이클을 **더 이상 도구 호출이 필요 없을 때까지 반복**한다.

```
Thought: ORD001의 상태를 확인해야 한다.
Action:  get_order_status("ORD001")
Observation: 배송 지연

Thought: ORD001은 배송 지연이다. ORD002도 확인하자.
Action:  get_order_status("ORD002")
Observation: 배송 완료

Thought: ORD002는 정상이다. ORD003도 확인하자.
Action:  get_order_status("ORD003")
Observation: 배송 중

Thought: 배송 지연은 ORD001뿐이다. 쿠폰을 발급하자.
Action:  issue_coupon("ORD001", 5000)
Observation: 쿠폰 발급 완료

Thought: 모든 작업이 완료되었다. 결과를 정리하자.
Final Answer: ORD001은 배송 지연으로 5,000원 쿠폰을 발급했고,
              ORD002(배송 완료), ORD003(배송 중)은 보상 대상이 아닙니다.
```

> **💡 ReAct가 혁신적인 이유**
>
> 기존 LLM은 "한 번에 모든 답변"을 생성하려 했다.
> ReAct는 `"한 단계씩 실행하고, 그 결과를 보고 다음 단계를 결정"`하는 방식이다.
> 이는 사람이 복잡한 문제를 풀 때의 사고 방식과 동일하다:
> 계획 → 실행 → 확인 → 다음 계획 → 실행 → ...

### 4-3. Direct vs ReAct 패턴

LangGraph에서 Agent의 도구 호출 방식은 크게 두 가지로 나뉜다.

| 패턴 | 그래프 흐름 | 도구 실행 후 | 적합한 상황 |
|------|-----------|:---:|------|
| **Direct** | agent → tools → **END** | 바로 종료 | 단순 1단계 요청 |
| **ReAct** | agent → tools → **agent** (순환) | 다시 추론 | 복잡한 다단계 요청 |

```
[Direct 패턴]  — 도구를 1회 실행하고 끝
START → [agent] → [tools] → END

[ReAct 패턴]  — 도구 실행 후 다시 agent로 돌아가 다음 행동을 결정
START → [agent] ⇄ [tools] → END
              ↑        │
              └────────┘  순환 엣지 (핵심!)
```

Direct 패턴에서 ReAct 패턴으로의 변경은 단 하나의 차이이다:
- Direct: `tools → END` (도구 실행 후 종료)
- ReAct: `tools → agent` (도구 실행 후 **다시 추론**)

> **💡 `create_agent`는 ReAct 패턴이 이미 내장되어 있다.**
>
> 앞에서 사용한 `create_agent`는 내부적으로 ReAct 순환 구조를 자동 생성한다.
> 별도의 설정 없이도 복잡한 다단계 요청을 처리할 수 있는 이유이다.
> 자기주도 실습(4-2_1)에서는 이 구조를 StateGraph로 **직접 구현**해 본다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: ReAct 패턴의 실제 동작 확인
# - 3개 주문을 확인하고 조건에 맞는 주문에만 쿠폰을 발급하는 다단계 요청
# - Agent가 내부적으로 여러 번의 Thought→Action→Observation을 반복한다.
# - issued_coupons로 "정확히 배송 지연 주문에만 발급했는지" 검증한다.
# ═══════════════════════════════════════════════════════════

# 쿠폰 기록 초기화 (이전 테스트 결과 제거)
issued_coupons.clear()

# 복잡한 다단계 요청: 3개 주문 확인 + 조건부 쿠폰 발급
complex_request = '''주문 ORD001, ORD002, ORD003의 상태를 모두 확인하고,
배송 지연인 주문에만 5000원 쿠폰을 발급해줘.'''

# ★ 핵심: 동일한 create_agent가 복잡한 요청도 처리한다.
# 내부적으로 ReAct 순환(agent ⇄ tools)이 여러 번 반복된다:
#   1회: get_order_status("ORD001") → 배송 지연
#   2회: get_order_status("ORD002") → 배송 완료
#   3회: get_order_status("ORD003") → 배송 중
#   4회: issue_coupon("ORD001", 5000) → 발급 완료
#   5회: 모든 결과를 종합하여 최종 답변 생성
memory_react = MemorySaver()
agent_react = create_agent(
    model=llm, tools=tools,
    system_prompt='당신은 고객 서비스 AI 에이전트입니다. 제공된 도구를 사용하여 고객 요청을 처리하세요.',
    checkpointer=memory_react,
)

result = agent_react.invoke(
    {'messages': [{'role': 'user', 'content': complex_request}]},
    config={'configurable': {'thread_id': 'react-test'}},
)

print('Agent 응답:')
print(result['messages'][-1].content)

# ★ 핵심 확인: ORD001(배송 지연)에만 쿠폰이 발급되었는가?
print(f'\n발급된 쿠폰: {issued_coupons}')
print('→ ORD001(배송 지연)에만 쿠폰이 발급되었다!')
print('  ORD002(배송 완료), ORD003(배송 중)은 건너뛰었다.')
print('  이것이 ReAct의 힘: 추론-행동을 반복하며 조건을 판단한다.')


---

## 5. Trustworthiness — Agent의 신뢰성 확보

### 5-1. Agent는 왜 위험할 수 있는가?

Agent는 Tool을 통해 **실제 시스템에 영향을 미친다** (쿠폰 발급, DB 수정 등).
따라서 잘못된 판단이 실제 피해로 이어질 수 있다.

| 위험 상황 | 예시 |
|---------|------|
| **정책 위반** | 한도를 초과하여 10만원 쿠폰 발급 |
| **조건 미충족** | 배송 완료된 주문에 배송 지연 보상 쿠폰 발급 |
| **유해 요청** | "시스템을 해킹해줘" 같은 악의적 요청 처리 |
| **민감 정보 노출** | 응답에 고객의 주민등록번호 포함 |

### 5-2. 3중 보호 구조

```
[사용자 입력] → [입력 가드레일] → [Agent/ReAct] → [검증 로직] → [도구 실행] → [출력 가드레일] → [응답]
                  ↑                                  ↑                         ↑
             유해 요청 차단              정책 준수 확인               민감 정보 필터링
```

| 보호 계층 | 역할 | 예시 |
|---------|------|------|
| **입력 가드레일** | 유해/부적절한 요청 차단 | "해킹", "비밀번호" 등 키워드 차단 |
| **검증 로직** | 도구 실행 전 정책 준수 확인 | 쿠폰 한도 체크, 주문 상태 확인 |
| **출력 가드레일** | 민감 정보 필터링 | 주민번호, 계좌번호 마스킹 |

<br>

> **💡 HITL (Human-in-the-Loop)**
>
> 고액 쿠폰 발급처럼 **고위험 행동**은 사람의 승인을 거치도록 설계할 수 있다.
> LangGraph의 `interrupt` 기능을 사용하면 특정 노드에서 실행을 일시 중지하고
> 사람의 확인을 받은 뒤 계속 진행할 수 있다.


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: 입력/출력 가드레일의 기본 구현
# - input_guardrail: 유해 키워드가 포함된 요청을 사전에 차단
# - output_guardrail: 응답에 포함된 민감 정보(주민번호 등)를 마스킹
# ═══════════════════════════════════════════════════════════

# ========== 입력 가드레일 ==========
# 역할: Agent 실행 전에 유해/부적절한 요청을 차단한다.
# 반환값: (통과 여부, 차단 메시지)
def input_guardrail(user_input: str) -> tuple[bool, str]:
    """입력 가드레일: 유해 요청을 차단한다."""
    blocked = ['해킹', '비밀번호', '탈취', '시스템 접근']
    for keyword in blocked:
        if keyword in user_input:
            return False, f'⚠️ 차단: "{keyword}"이 포함된 요청은 처리할 수 없습니다.'
    return True, ''  # 통과

# ========== 출력 가드레일 ==========
# 역할: Agent 응답에서 민감 정보를 제거/마스킹한다.
# ★ 핵심: re.sub()로 정규표현식 패턴에 매칭되는 부분을 마스킹 처리
def output_guardrail(response: str) -> str:
    """출력 가드레일: 민감 정보를 마스킹한다."""
    import re
    response = re.sub(r'\d{6}-\d{7}', '******-*******', response)  # 주민번호 패턴
    return response

# ========== 테스트 ==========
print('입력 가드레일 테스트:')
ok1, msg1 = input_guardrail('주문 ORD001 상태 확인해줘')
print(f'  정상 요청: 통과={ok1}')

ok2, msg2 = input_guardrail('시스템 접근 권한을 줘')
print(f'  유해 요청: 통과={ok2}, {msg2}')

print('\n출력 가드레일 테스트:')
masked = output_guardrail('고객님의 주민번호는 901215-1234567입니다.')
print(f'  마스킹 결과: {masked}')


---

## 6. Multi-Agent 패턴

![Image C](https://i.ibb.co/jZ5MRpGm/image-C.png)

### 6-1. 왜 Multi-Agent가 필요한가?

지금까지 만든 Agent는 **하나의 LLM이 모든 것을 처리**하는 구조였다.
단순한 작업에서는 충분하지만, 복잡한 작업에서는 한계가 드러난다.

```
[단일 Agent에게 복잡한 작업을 맡기면]

"3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산도 포함해서."

→ 하나의 LLM이 동시에 처리해야 할 것:
  - 일정 계획 (여행 전문가 역할)
  - 맛집 추천 (음식 전문가 역할)
  - 관광지 추천 (관광 전문가 역할)
  - 예산 계산 (회계 역할)

→ 결과: 일부 항목 누락, 깊이 부족, 체계 없는 답변
```

> **💡 비유: 1인 식당 vs 전문 레스토랑**
>
> - **단일 Agent** = 1인 식당. 한 명이 주문, 요리, 서빙, 계산을 모두 담당 → 바쁘면 품질 저하
> - **Multi-Agent** = 전문 레스토랑. 셰프는 요리, 웨이터는 서빙, 매니저는 관리 → 역할 분담으로 품질 향상

### 6-2. Multi-Agent의 핵심: 역할 분리 + 상태 공유

Multi-Agent 시스템의 두 가지 핵심 원리:

1. **역할 분리**: 각 Agent가 하나의 전문 역할만 담당한다
   - Planner Agent: 계획만 수립 (실행하지 않음)
   - Worker Agent: 계획을 실행 (계획하지 않음)
   - Reflection Agent: 결과를 검토 (실행하지 않음)

2. **상태 공유 (State)**: LangGraph의 State를 통해 Agent 간에 정보를 주고받는다
   - Planner가 `plan` 필드에 계획을 기록
   - Worker가 `plan`을 읽고 실행, `result` 필드에 결과 기록
   - Reflection이 `result`를 읽고 검토

### 6-3. 대표 Multi-Agent 패턴 5가지

| 패턴 | 구조 | 핵심 원리 | 적합한 상황 |
|------|------|---------|------|
| **Planner-Worker** | 계획 → 실행 | 계획과 실행의 분리 | 다단계 작업 (여행 계획 등) |
| **Supervisor-Worker** | 관리자 → 전문가들 | 중앙 관리자가 작업 분배 | 전문 영역이 다른 작업 |
| **Reflection** | 실행 → 검토 → 개선 | 자기 검토 루프 | 품질이 중요한 작업 |
| **Debate** | Agent A ⇄ Agent B | 다양한 관점의 토론 | 의사결정, 분석 |
| **Pipeline** | A → B → C → D | 순차적 가공 | 데이터 처리 파이프라인 |

이 실습에서는 가장 기본인 **Planner-Worker**와 품질 향상을 위한 **Reflection**을 직접 구현해 본다.


### 6-4. Planner-Worker 패턴

가장 기본적이면서 강력한 Multi-Agent 패턴이다.

```
START → [Planner] → [Worker] → END
          계획 수립     계획 실행
```

**핵심 원칙: "계획하는 자와 실행하는 자를 분리하라."**

| Agent | 역할 | 프롬프트 핵심 지시 |
|-------|------|------------------|
| **Planner** | 요청을 분석하고 Step 1, 2, 3... 형태의 계획 수립 | "계획만 수립하세요. **직접 실행하지 마세요.**" |
| **Worker** | Planner의 계획을 받아 각 Step을 실행 | "계획대로 실행하고 구체적인 결과를 작성하세요." |

> **💡 왜 분리하면 품질이 올라가는가?**
>
> 하나의 LLM에게 "계획도 세우고 실행도 해"라고 하면,
> 계획이 부실하거나 계획을 세우다가 실행으로 넘어가 버리는 경우가 많다.
> Planner에게 `"실행하지 마"`라고 명시하면, 계획 수립에만 집중하여 더 체계적인 계획이 나온다.
> Worker는 이미 만들어진 계획을 따르기만 하면 되므로 실행 품질도 높아진다.


In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Planner-Worker Multi-Agent 워크플로우 구현
# 1. State 정의: Agent 간에 공유되는 데이터 구조
# 2. Planner Node: 계획만 수립 (실행 안 함)
# 3. Worker Node: 계획을 받아 실행
# 4. StateGraph로 연결: planner → worker → END
# ═══════════════════════════════════════════════════════════

# ========== 1. State 정의 ==========
# 역할: 모든 Agent가 읽고 쓰는 공유 데이터 구조 ("칠판")
# 각 Agent는 자기 담당 필드만 업데이트한다.
class MultiAgentState(TypedDict):
    user_request: str     # 사용자 요청 (입력)
    plan: str             # Planner가 기록 → Worker가 읽음
    result: str           # Worker가 기록 → Reflection이 읽음
    reflection: str       # Reflection이 기록
    final_result: str     # 최종 결과
    reflection_count: int # 무한 루프 방지 카운터

# ========== 2. Planner Node ==========
# 역할: 사용자 요청을 분석하고 "Step 1, 2, 3..." 형태의 계획을 수립한다.
# ★ 핵심: 프롬프트에 "계획만 수립하세요. 직접 실행하지 마세요."를 명시한다.
#   이렇게 해야 Planner가 계획 수립에만 집중하고, 실행은 Worker에게 맡긴다.
def planner_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f'''당신은 작업 계획을 수립하는 Planner Agent입니다.
사용자 요청을 분석하고, 수행해야 할 작업을 단계별로 나열하세요.
계획만 수립하세요. 직접 실행하지 마세요.

출력 형식:
Step 1: [작업 설명]
Step 2: [작업 설명]
...

사용자 요청: {state["user_request"]}'''
    response = llm.invoke(prompt)
    return {'plan': response.content}  # State의 'plan' 필드만 업데이트

# ========== 3. Worker Node ==========
# 역할: Planner의 계획(state['plan'])을 읽어 각 Step을 실행한다.
# ★ 핵심: state['plan']에서 계획을 가져와 프롬프트에 포함시킨다.
#   Worker는 계획을 세우지 않고, 주어진 계획을 실행하는 데만 집중한다.
def worker_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f'''당신은 계획을 실행하는 Worker Agent입니다.
Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요.

원본 요청: {state["user_request"]}
실행할 계획:
{state["plan"]}

각 단계별 실행 결과를 구체적으로 작성하세요.'''
    response = llm.invoke(prompt)
    return {'result': response.content, 'final_result': response.content}

# ========== 4. StateGraph로 워크플로우 구성 ==========
# 4-1에서 배운 LangGraph 4단계 패턴과 동일하다:
# State 정의 → Node 정의 → Graph 구성 → 컴파일
workflow_pw = StateGraph(MultiAgentState)
workflow_pw.add_node('planner', planner_node)  # 노드 등록
workflow_pw.add_node('worker', worker_node)
workflow_pw.add_edge(START, 'planner')          # 시작 → Planner
workflow_pw.add_edge('planner', 'worker')        # Planner → Worker
workflow_pw.add_edge('worker', END)              # Worker → 종료

app_pw = workflow_pw.compile()  # 컴파일: 실행 가능한 그래프로 변환
print('Planner-Worker 워크플로우 구성 완료!')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Planner-Worker 실행 결과 확인
# - Planner가 계획을 수립하고, Worker가 실행하는 분업을 확인한다.
# - result['plan']과 result['result']를 각각 확인한다.
# ═══════════════════════════════════════════════════════════

# invoke()에 초기 State를 전달하면 planner → worker 순서로 실행된다.
result = app_pw.invoke({
    'user_request': '3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산도 포함해서.',
    'reflection_count': 0,
})

# Planner가 수립한 계획 확인
print('=== Planner의 계획 ===')
print(result['plan'][:500])

# Worker가 실행한 결과 확인
print('\n=== Worker의 실행 결과 ===')
print(result['result'][:500])


### 6-5. Reflection 패턴 — 자기 검토를 통한 품질 향상

![Image D](https://i.ibb.co/Fqjv26fX/image-D.png)

Planner-Worker만으로도 결과를 얻을 수 있지만, **한 번에 완벽한 결과가 나오지 않을 수 있다.**
Reflection 패턴은 **결과를 검토하고, 미달이면 개선하는 루프**를 추가한다.

```
START → [Planner] → [Worker] → [Reflection] ──→ END (품질 충족)
                        ↑            │      
                        └────────────┘  (품질 미달 → Worker 재실행)
```

**Reflection의 핵심 동작:**

| 단계 | 설명 |
|------|------|
| 1. **검토** | Worker의 결과가 원래 요청의 모든 항목을 충족하는지 확인 |
| 2. **판정** | "충족" 또는 "미충족" 판단 |
| 3. **피드백** | 미충족 시 "어떤 부분이 부족한지" 구체적으로 명시 |
| 4. **재실행** | 피드백과 함께 Worker를 다시 실행 |

<br>

> **💡 Reflection에서 가장 중요한 설계 포인트: 무한 루프 방지**
>
> Reflection이 계속 "미충족"을 반환하면 Worker가 무한히 재실행된다.
> 따라서 반드시 **`reflection_count`**로 최대 반복 횟수를 제한해야 한다.
> 이 실습에서는 `MAX_REFLECTIONS = 1`로 설정하여 최대 1회 재실행만 허용한다.

**LangGraph에서의 구현:**
- `add_conditional_edges`를 사용하여 Reflection 결과에 따라 분기
- 품질 충족 → `END`로 이동
- 품질 미달 → `worker` 노드로 다시 이동 (조건부 순환)


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Reflection 패턴 구현
# 1. reflection_node: Worker 결과를 검토하고 "충족/미충족" 판정
# 2. should_continue: 판정 결과에 따라 END 또는 Worker 재실행
# 3. add_conditional_edges: 조건부 분기로 순환 구조 구현
# ═══════════════════════════════════════════════════════════

# ★ 무한 루프 방지: 최대 재실행 횟수를 제한한다.
MAX_REFLECTIONS = 1

# ========== Reflection Node ==========
# 역할: Worker의 결과(state['result'])를 검토한다.
# - "충족" 판정 → final_result에 결과를 기록 (종료 신호)
# - "미충족" 판정 → final_result를 기록하지 않음 (Worker 재실행 신호)
def reflection_node(state: MultiAgentState) -> MultiAgentState:
    """Worker의 결과를 검토하고, 개선이 필요하면 피드백을 준다."""
    count = state.get('reflection_count', 0)

    prompt = f'''당신은 품질 검토 Agent입니다.
Worker의 결과를 검토하고, 다음 형식으로 평가하세요.

원본 요청: {state["user_request"]}
Worker 결과:
{state["result"]}

평가 기준:
1. 요청의 모든 항목이 포함되었는가?
2. 정보가 구체적인가?

마지막에 반드시 "충족" 또는 "미충족"으로 판정하세요.'''
    response = llm.invoke(prompt)
    reflection = response.content

    # ★ 핵심: "충족" 판정 시에만 final_result를 기록한다.
    # should_continue 함수가 이 값의 유무로 종료 여부를 결정한다.
    if '충족' in reflection and '미충족' not in reflection:
        return {
            'reflection': reflection,
            'final_result': state['result'],  # ← 종료 신호
            'reflection_count': count + 1,
        }
    else:
        return {
            'reflection': reflection,
            'reflection_count': count + 1,
            # final_result를 설정하지 않음 → Worker 재실행
        }

# ========== 분기 함수 ==========
# 역할: Reflection 결과에 따라 다음 행선지를 결정한다.
# ★ 핵심: 반환값이 add_conditional_edges의 매핑 키와 일치해야 한다.
def should_continue(state: MultiAgentState) -> str:
    """Reflection 결과에 따라 Worker 재실행 여부를 결정한다."""
    if state.get('reflection_count', 0) >= MAX_REFLECTIONS + 1:
        return 'end'     # 최대 횟수 초과 → 강제 종료
    if state.get('final_result'):
        return 'end'     # 품질 충족 → 정상 종료
    return 'worker'      # 미충족 → Worker 재실행

# ========== Reflection 워크플로우 구성 ==========
workflow_ref = StateGraph(MultiAgentState)
workflow_ref.add_node('planner', planner_node)
workflow_ref.add_node('worker', worker_node)
workflow_ref.add_node('reflection', reflection_node)

workflow_ref.add_edge(START, 'planner')       # 시작 → Planner
workflow_ref.add_edge('planner', 'worker')     # Planner → Worker
workflow_ref.add_edge('worker', 'reflection')  # Worker → Reflection

# ★ 핵심 코드: add_conditional_edges()
# Reflection 노드 실행 후, should_continue 함수의 반환값에 따라 분기한다.
# - 'end' 반환 → END로 이동 (종료)
# - 'worker' 반환 → worker 노드로 이동 (재실행)
workflow_ref.add_conditional_edges(
    'reflection',       # 출발 노드
    should_continue,    # 분기 판단 함수
    {'end': END, 'worker': 'worker'},  # 반환값 → 노드 매핑
)

app_ref = workflow_ref.compile()
print('Reflection 워크플로우 구성 완료!')
print('흐름: planner → worker → reflection → (END 또는 worker 재실행)')


In [ ]:
# ═══════════════════════════════════════════════════════════
# 이 셀의 핵심: Reflection 패턴 실행 결과 확인
# - reflection_count: Reflection이 몇 회 실행되었는지
# - reflection: 검토 내용 (충족/미충족 판정 + 피드백)
# - final_result: 최종 결과 (충족 시 기록됨)
# ═══════════════════════════════════════════════════════════

result_ref = app_ref.invoke({
    'user_request': '3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산도 포함해서.',
    'reflection_count': 0,
})

# Reflection Agent의 검토 결과
print('=== Reflection 검토 결과 ===')
print(result_ref.get('reflection', '')[:400])

# Reflection 반복 횟수 (MAX_REFLECTIONS=1이므로 최대 2회)
print(f'\nReflection 횟수: {result_ref.get("reflection_count", 0)}회')

# 최종 결과 (충족 시 final_result, 미충족 시 result)
print(f'\n=== 최종 결과 (앞 500자) ===')
print(result_ref.get('final_result', result_ref.get('result', ''))[:500])


---

## 7. 정리

### 오늘 배운 전체 흐름

```
① LLM만 → 텍스트만 생성, 실제 행동 불가 (챕터 2)
② +Tool → LLM이 외부 함수를 호출하여 실제 행동 가능 (챕터 2)
③ +Memory → 대화 맥락 유지 (챕터 3)
④ +Plan(ReAct) → 복잡한 다단계 요청 처리 (챕터 4)
⑤ +Trustworthiness → 가드레일과 검증으로 안전성 확보 (챕터 5)
⑥ Multi-Agent → 역할 분담으로 품질 향상 (챕터 6)
```

### 핵심 개념 요약

| 개념 | 한 줄 정리 |
|------|----------|
| **Agent** | LLM + Tool + Memory + Plan — LLM이 스스로 판단하고 행동 |
| **Tool-use** | LLM이 외부 함수를 호출하여 실제 시스템에 영향을 미침 |
| **Memory** | `MemorySaver` + `thread_id`로 대화 맥락 유지 |
| **ReAct** | Thought → Action → Observation 반복으로 복잡한 작업 처리 |
| **Direct vs ReAct** | 단순 요청은 Direct, 복잡한 다단계 요청은 ReAct |
| **Trustworthiness** | 입력 가드레일 + 검증 로직 + 출력 가드레일의 3중 보호 |
| **Planner-Worker** | 계획 수립과 실행을 분리하는 Multi-Agent 패턴 |
| **Reflection** | 결과를 검토하고 미달 시 재실행하는 품질 향상 패턴 |

### RAG(4-1) → Agent(4-2) 관계 정리

| 구분 | RAG (4-1) | Agent (4-2) |
|------|:---:|:---:|
| 핵심 | 검색 + 생성 | **판단 + 행동** |
| 흐름 | 고정 (retrieve → generate) | **LLM이 동적으로 결정** |
| 도구 | Retriever만 | 다양한 도구 (검색, 쿠폰 발급, DB 조회 등) |
| 결합 | - | RAG를 Tool로 Agent에 통합 가능 |

### 실습 안내

- `실습_4-2_1`: Agent의 4대 구성요소를 하나씩 추가하며 ReAct Agent를 완성한다.
- `실습_4-2_2`: Planner-Worker + Reflection 패턴의 Multi-Agent 시스템을 구현한다.

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.
